In [ ]:
# These are default values. Papermill will overwrite these at runtime.
question = "What are the prerequisites for CPT S 223?"
student_context = {"major": "Computer Science", "completed_courses": []}
use_rag = True

In [ ]:
import os
import time

# Start timer
start_time = time.time()

base_dir = os.getcwd()

db_path = os.path.join(base_dir, "data", "courses.db")
model_path = os.path.join(base_dir, "data", "models", "llama-3.1-8b-q4.gguf")

index_path = os.path.join(base_dir, "prompt-search", "data", "domain", "courses.faiss")
meta_path = os.path.join(base_dir, "prompt-search", "data", "domain", "metadata.json")

# Verification print (shows up in terminal if fail)
if not os.path.exists(model_path):
    print(f"CRITICAL ERROR: Model not found at {model_path}")
if not os.path.exists(db_path):
    print(f"CRITICAL ERROR: Database not found at {db_path}")

In [ ]:
import os
import json
import time
import sqlite3
import re
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama

start_time = time.time()
db_path = os.path.join(base_dir, "data", "courses.db")
model_path = os.path.join(base_dir, "data", "models", "llama-3.1-8b-q4.gguf")
index_path = os.path.join(base_dir, "prompt-search", "data", "domain", "courses.faiss")
meta_path = os.path.join(base_dir, "prompt-search", "data", "domain", "metadata.json")

# RECURSIVE SQLITE LOGIC
def fetch_course_info(prefix, number):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    clean_prefix = prefix.upper().replace(' ', '')
    cursor.execute("SELECT prefix, courseNumber, title, coursePrerequisite FROM courses WHERE REPLACE(UPPER(prefix), ' ', '') = ? AND courseNumber = ? LIMIT 1", (clean_prefix, number))
    row = cursor.fetchone()
    conn.close()
    return row

def build_prereq_chain(query, visited=None):
    if visited is None: visited = set()
    match = re.search(r'([A-Z]{2,4}(?:\sS)?)\s*(\d{3})', query.upper())
    if not match: return []
    prefix, number = match.groups()
    course_id = f"{prefix.strip()} {number}"
    
    if course_id in visited: return []
    visited.add(course_id)
    
    row = fetch_course_info(prefix, number)
    if not row: return []
    
    chain = [{"code": f"{row[0]} {row[1]}", "title": row[2], "prereqs": row[3] or "None"}]
    if row[3] and row[3] != "None":
        found_codes = re.findall(r'([A-Z]{2,4}(?:\sS)?\s*\d{3})', row[3].upper())
        for code in found_codes:
            chain.extend(build_prereq_chain(code, visited))
    return chain

# FAISS LOGIC
def retrieve_context(query, top_k=3):
    index = faiss.read_index(index_path)
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    with open(meta_path, 'r', encoding='utf-8') as f: metadata = json.load(f)
    query_vector = encoder.encode([query])
    distances, indices = index.search(np.array(query_vector).astype('float32'), top_k)
    results = []
    for i in indices[0]:
        if i != -1:
            chunk = metadata[i]
            results.append(f"Course: {chunk.get('course_code', 'Unknown')}\nPrereqs: {chunk.get('prereq_raw', 'None')}\n{chunk.get('chunk_text', '')}")
    return "\n\n".join(results)

# ROUTING LOGIC 
is_classification = "Answer ONLY with YES or NO" in question
is_chain_request = any(word in question.lower() for word in ['path', 'chain', 'sequence', 'steps', 'prerequisite']) and not is_classification
use_rag = True

if is_classification:
    system_message = "You are a classification assistant. Answer ONLY with YES or NO."
    context_str = "No context needed."
    sources = []
elif is_chain_request:
    system_message = "You are a WSU advisor. Summarize the prerequisite path concisely. Do not repeat instructions."
    chain_data = build_prereq_chain(question)
    context_str = "PREREQUISITES:\n" + "\n".join([f"- {c['code']}: needs {c['prereqs']}" for c in chain_data])
    sources = [c['code'] for c in chain_data]
else:
    system_message = "You are a WSU Academic Advisor. Answer concisely based on the context."
    context_str = retrieve_context(question)
    sources = ["FAISS Search"]

# LLM INFERENCE 
# n_gpu_layers=-1 offloads to GPU for maximum speed
llm = Llama(model_path=model_path, n_ctx=4096, verbose=False, n_gpu_layers=-1)
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {question}"}
    ],
    temperature=0.1,
    max_tokens=500
)
answer = response['choices'][0]['message']['content'].strip()

# EXPORT 
output = {
    "answer": answer,
    "sources": sources,
    "metadata": {
        "used_rag": use_rag,
        "latency_seconds": round(time.time() - start_time, 2)
    }
}

temp_path = os.path.join(os.environ.get('TEMP', '/tmp'), 'llm_response.json')
with open(temp_path, 'w') as f:
    json.dump(output, f)
print(json.dumps(output))